# Linear Regression Model Training/Validation

In [39]:

%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import KFold, cross_val_score, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np
import pickle

from helpers.plotting import *

DATA_PATH = "./data/"
MODELS_PATH = "./models/"
SUBMISSIONS_PATH = "./submissions/"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Here we shall train and validate different iterations of a Linear Regression Model based on the resulting dataset of the preprocessing notebook

In [20]:
rmse = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

## Baseline Linear model

Premise:
- One hot encoded variables
- Dropped episode title
- Outliers removed from Episode Length and N of Ads
- Missing values imputed with random sample means of 1% of the data (~7500 samples)

In [41]:
training_data = pl.read_csv(DATA_PATH + "one_hot_encoded_for_lr.csv")
test_data = pl.read_csv(DATA_PATH + "test.csv")

In [36]:
numerical_cols = ["Host_Popularity_percentage", "Guest_Popularity_percentage", "Number_of_Ads", "Episode_Length_minutes"]

lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
    ],
    remainder="passthrough"  # Keeps the other columns as-is
)

In [37]:
X = training_data.drop("Listening_Time_minutes")
y = training_data["Listening_Time_minutes"]

baseline_lr_pipeline = Pipeline([
    ("preprocessor", lr_preprocessor),
    ("model", LinearRegression())
])

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error"
}

cv_baseline_lr_res = cross_validate(baseline_lr_pipeline, X, y, cv=kfold, scoring=scoring, n_jobs=-1, return_estimator=True)

rmse_scores_baseline = -cv_baseline_lr_res["test_rmse"]
r2_scores_baseline = cv_baseline_lr_res["test_r2"]

print("Average R²:", r2_scores_baseline.mean())
print("Average RMSE:", rmse_scores_baseline.mean())

Average R²: 0.7574259186331247
Average RMSE: 13.365936781340116


In [40]:
baseline_lr_pipeline_file = MODELS_PATH + "baseline_lr_pipeline.pkl"
with open(baseline_lr_pipeline_file, 'wb') as file:
    pickle.dump(baseline_lr_pipeline, file)

In [42]:
def preprocessing_for_lr(df: pl.DataFrame) -> pl.DataFrame:
    dropped_id_and_title = df.drop("id", "Episode_Title")
    one_hot_encoded = dropped_id_and_title \
        .to_dummies("Podcast_Name") \
        .to_dummies("Genre") \
        .to_dummies("Publication_Day") \
        .to_dummies("Publication_Time") \
        .to_dummies("Episode_Sentiment")
    
    return one_hot_encoded

In [43]:
prepared_test = preprocessing_for_lr(test_data)

In [46]:
baseline_lr_pipeline.fit(X, y)

/Users/luiscruz/Desktop/kaggle_challenges/regression/podcast-listening-time/.venv/lib/python3.12/site-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Host_Popularity_percentage',
                                                   'Guest_Popularity_percentage',
                                                   'Number_of_Ads',
                                                   'Episode_Length_minutes'])])),
                ('model', LinearRegression())])

In [48]:
prepared_test.null_count()

Podcast_Name_Athlete's Arena,Podcast_Name_Brain Boost,Podcast_Name_Business Briefs,Podcast_Name_Business Insights,Podcast_Name_Comedy Corner,Podcast_Name_Crime Chronicles,Podcast_Name_Criminal Minds,Podcast_Name_Current Affairs,Podcast_Name_Daily Digest,Podcast_Name_Detective Diaries,Podcast_Name_Digital Digest,Podcast_Name_Educational Nuggets,Podcast_Name_Fashion Forward,Podcast_Name_Finance Focus,Podcast_Name_Fitness First,Podcast_Name_Funny Folks,Podcast_Name_Gadget Geek,Podcast_Name_Game Day,Podcast_Name_Global News,Podcast_Name_Health Hour,Podcast_Name_Healthy Living,Podcast_Name_Home & Living,Podcast_Name_Humor Hub,Podcast_Name_Innovators,Podcast_Name_Joke Junction,Podcast_Name_Laugh Line,Podcast_Name_Learning Lab,Podcast_Name_Life Lessons,Podcast_Name_Lifestyle Lounge,Podcast_Name_Market Masters,Podcast_Name_Melody Mix,Podcast_Name_Mind & Body,Podcast_Name_Money Matters,Podcast_Name_Music Matters,Podcast_Name_Mystery Matters,Podcast_Name_News Roundup,Podcast_Name_Sound Waves,…,Podcast_Name_Sports Weekly,Podcast_Name_Study Sessions,Podcast_Name_Style Guide,Podcast_Name_Tech Talks,Podcast_Name_Tech Trends,Podcast_Name_True Crime Stories,Podcast_Name_Tune Time,Podcast_Name_Wellness Wave,Podcast_Name_World Watch,Episode_Length_minutes,Genre_Business,Genre_Comedy,Genre_Education,Genre_Health,Genre_Lifestyle,Genre_Music,Genre_News,Genre_Sports,Genre_Technology,Genre_True Crime,Host_Popularity_percentage,Publication_Day_Friday,Publication_Day_Monday,Publication_Day_Saturday,Publication_Day_Sunday,Publication_Day_Thursday,Publication_Day_Tuesday,Publication_Day_Wednesday,Publication_Time_Afternoon,Publication_Time_Evening,Publication_Time_Morning,Publication_Time_Night,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment_Negative,Episode_Sentiment_Neutral,Episode_Sentiment_Positive
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,28736,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,48832,0,0,0,0


In [47]:
y_pred_baseline_lr = baseline_lr_pipeline.predict(prepared_test)

baseline_lr_submission = pl.DataFrame({
    "id": test_data["id"],
    "Listening_Time_minutes": y_pred_baseline_lr
})

baseline_lr_submission

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## Lasso Regularization

Lasso Regularization avoids overfitting by shrinking some coefficients to zero

In [38]:
lasso_lr_pipeline = Pipeline([
    ("preprocessor", lr_preprocessor),
    ("model", Lasso(max_iter=10000))
])

cv_lasso_res = cross_validate(lasso_lr_pipeline, X, y, cv=kfold, scoring=scoring, n_jobs=-1, return_estimator=True)

rmse_scores_lasso = -cv_lasso_res["test_rmse"]
r2_scores_lasso = cv_lasso_res["test_r2"]

print("Average R²:", r2_scores_lasso.mean())
print("Average RMSE:", rmse_scores_lasso.mean())

Average R²: 0.7531975275698367
Average RMSE: 13.481927922965923


## Lasso w/ Hyperparameter Tuning

In [32]:
# Define hyperparameter grid
param_grid = {
    "model__alpha": np.logspace(-4, 1, 10)  # 10 numbers evenly spaced from 10 ** -4 = 0.0001 to 10 ** 1 = 10
}

# Set up GridSearchCV
grid_search_lasso = GridSearchCV(
    estimator=lasso_lr_pipeline,
    param_grid=param_grid,
    cv=kfold,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    return_train_score=True,
)

# Run grid search
grid_search_lasso.fit(X, y)

# Best results
best_alpha = grid_search_lasso.best_params_["model__alpha"]
best_rmse = -grid_search_lasso.best_score_
best_model = grid_search_lasso.best_estimator_

print(f"Best alpha: {best_alpha}")
print(f"Best RMSE: {best_rmse:.4f}")
print("Best R² on full data:", best_model.score(X, y))

/Users/luiscruz/Desktop/kaggle_challenges/regression/podcast-listening-time/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best alpha: 0.00035938136638046257
Best RMSE: 13.3658
Best R² on full data: 0.7574740961744758


The Lasso regression with the optimal alpha found in this search space is not much better than a regular baseline linear regression